# Vending-Bench 2: Teaching an LLM to Run a Vending Machine Business with GRPO

This notebook trains a small LLM using **GRPO** (Group Relative Policy Optimization) to make daily business decisions in the **Vending-Bench 2** environment.

**VB2** is a 365-day vending machine simulation where the agent must:
- Set prices for 5 products (soda, water, candy, chips, sandwiches)
- Manage inventory through adversarial supplier negotiations
- Handle weather, seasonal demand, and customer complaints
- Maximize final bank balance (sparse reward)

**Training approach:** We train the model to output JSON pricing/ordering decisions for each day. The reward is the change in bank balance after executing those decisions.

**Requirements:** Free Colab T4 GPU. Uses Unsloth for memory-efficient LoRA training.

In [ ]:
%%capture
!pip install -qqq unsloth vllm
!pip install -qqq trl transformers datasets accelerate
!pip install -qqq openenv-core fastmcp matplotlib

In [ ]:
%%capture
# Clone the VB2 environment code
!git clone https://github.com/retroam/vendsim-vb2.git /tmp/vendsim-vb2 2>/dev/null || true
import sys
sys.path.insert(0, '/tmp/vendsim-vb2')

## 1. Load Model with Unsloth

We use Qwen2.5-1.5B-Instruct with 4-bit quantization + LoRA for efficient training on a T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## 2. VB2 Environment Setup

We use the local `VendingBench2Environment` directly — no server needed for training.

In [ ]:
from vendsim_vb2.environment import VendingBench2Environment
from vendsim_vb2.demand import PRODUCTS
import json, random

# Quick sanity check
env = VendingBench2Environment(seed=42)
env.reset()
print(f"Starting balance: ${env.state.cash_balance}")
print(f"Products: {list(PRODUCTS.keys())}")
print(f"Ideal prices: {[(p, PRODUCTS[p]['ideal_price']) for p in PRODUCTS]}")

# Run a few days to show mechanics
for day in range(5):
    result = env.wait_for_next_day()
    print(f"Day {env.state.day_index-1}: sales={result.payload['sales']}, revenue=${result.payload['revenue']}, balance=${env.state.cash_balance}")

## 3. Define Reward Functions

We use three reward signals:
1. **`reward_revenue`**: Daily revenue from the pricing decisions (dense signal)
2. **`reward_valid_json`**: Whether the model output valid, parseable JSON
3. **`reward_balance_delta`**: Change in bank balance after executing decisions

In [ ]:
PRODUCT_NAMES = list(PRODUCTS.keys())

VB2_PROMPT_TEMPLATE = """You are managing a vending machine business. Make pricing decisions to maximize profit.

Current state:
- Day: {day}/365
- Balance: ${balance:.2f}
- Weather: {weather}
- Season: {season}
- Machine inventory: {machine_inv}
- Yesterday's sales: {last_sales}

Products and their current prices:
{price_list}

Output a JSON object with your pricing decisions. Set prices to maximize revenue.
Format: {{"soda": 1.50, "water": 1.25, "candy": 1.25, "chips": 2.00, "sandwich": 4.50}}
Only output the JSON, nothing else."""


def parse_pricing_decision(text):
    """Extract JSON pricing from model output."""
    text = text.strip()
    # Try to find JSON in the text
    for start_char in ['{', '```']:
        idx = text.find('{')
        if idx >= 0:
            end = text.find('}', idx)
            if end >= 0:
                try:
                    return json.loads(text[idx:end+1])
                except json.JSONDecodeError:
                    pass
    return None


def evaluate_pricing_decision(completion_text, seed=None):
    """Run a single day in VB2 with the given pricing, return metrics."""
    pricing = parse_pricing_decision(completion_text)
    env = VendingBench2Environment(seed=seed or random.randint(0, 100000))
    env.reset()
    
    # Stock the machine with some initial inventory
    for product in PRODUCT_NAMES:
        env.state.storage_inventory[product] = 20
        env.run_sub_agent("restock_machine", product=product, qty=3)
    
    balance_before = env.state.cash_balance
    valid_json = pricing is not None
    
    if valid_json:
        for product, price in pricing.items():
            if product in PRODUCT_NAMES and isinstance(price, (int, float)):
                env.set_price(product, max(0.01, min(price, 20.0)))
    
    result = env.wait_for_next_day()
    balance_after = env.state.cash_balance
    revenue = result.payload.get('revenue', 0.0)
    balance_delta = balance_after - balance_before
    
    return {
        'valid_json': 1.0 if valid_json else -1.0,
        'revenue': min(revenue / 20.0, 2.0),  # Normalize to ~[0, 2]
        'balance_delta': balance_delta / 10.0,  # Normalize
    }


# Test it
test_result = evaluate_pricing_decision('{"soda": 1.50, "water": 1.25, "candy": 1.00, "chips": 2.50, "sandwich": 4.00}')
print(f"Test evaluation: {test_result}")

## 4. Build Training Dataset

Generate diverse prompts from different VB2 states (different days, weather, balances).

In [ ]:
from datasets import Dataset
from vendsim_vb2.demand import season_for_day, weather_for_day, day_of_week_for_day

def generate_prompts(n=256):
    prompts = []
    for i in range(n):
        day = random.randint(1, 365)
        balance = 500.0 + random.uniform(-200, 300)
        weather = weather_for_day(day)
        season = season_for_day(day)
        
        # Random recent sales history
        last_sales = {p: random.randint(0, 8) for p in PRODUCT_NAMES}
        machine_inv = {p: random.randint(0, 6) for p in PRODUCT_NAMES}
        
        price_list = "\n".join(
            f"  {p}: ${PRODUCTS[p]['ideal_price']:.2f} (wholesale: ${PRODUCTS[p]['wholesale_price']:.2f})"
            for p in PRODUCT_NAMES
        )
        
        prompt = VB2_PROMPT_TEMPLATE.format(
            day=day,
            balance=balance,
            weather=weather,
            season=season,
            machine_inv=json.dumps(machine_inv),
            last_sales=json.dumps(last_sales),
            price_list=price_list,
        )
        prompts.append(prompt)
    return prompts

prompts = generate_prompts(256)
dataset = Dataset.from_dict({"prompt": prompts})
print(f"Dataset size: {len(dataset)}")
print(f"\nExample prompt:\n{prompts[0][:500]}...")

## 5. Reward Functions for GRPO

These receive completions and return scalar rewards.

In [ ]:
def reward_valid_json(completions, **kwargs):
    """Reward for producing valid JSON output."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        parsed = parse_pricing_decision(text)
        if parsed is None:
            rewards.append(-1.0)
        elif all(p in parsed for p in PRODUCT_NAMES):
            rewards.append(1.0)  # All products priced
        else:
            rewards.append(0.3)  # Partial pricing
    return rewards


def reward_revenue(completions, **kwargs):
    """Reward based on simulated daily revenue."""
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        try:
            metrics = evaluate_pricing_decision(text, seed=i)
            rewards.append(metrics['revenue'])
        except Exception:
            rewards.append(0.0)
    return rewards


def reward_balance(completions, **kwargs):
    """Reward based on balance improvement."""
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        try:
            metrics = evaluate_pricing_decision(text, seed=i)
            rewards.append(metrics['balance_delta'])
        except Exception:
            rewards.append(-0.5)
    return rewards


print("Reward functions defined.")

## 6. Configure and Run GRPO Training

We train with 3 reward signals to guide the model toward valid, profitable pricing decisions.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="vb2_grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=128,
    max_prompt_length=max_seq_length - 128,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    max_steps=100,
    temperature=0.8,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_valid_json,
        reward_revenue,
        reward_balance,
    ],
    args=training_args,
    train_dataset=dataset,
)

print("Trainer initialized. Starting training...")

In [ ]:
trainer.train()
print("Training complete!")

## 7. Visualize Reward Curves

Plot how rewards improve over training steps.

In [ ]:
import matplotlib.pyplot as plt

# Extract training history
log_history = trainer.state.log_history
steps = [entry['step'] for entry in log_history if 'reward' in entry]
rewards = [entry['reward'] for entry in log_history if 'reward' in entry]
losses = [entry['loss'] for entry in log_history if 'loss' in entry]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
axes[0].plot(steps, rewards, 'b-', alpha=0.7, linewidth=1)
if len(rewards) >= 10:
    window = min(10, len(rewards))
    smoothed = [sum(rewards[max(0,i-window):i+1])/min(i+1,window) for i in range(len(rewards))]
    axes[0].plot(steps, smoothed, 'r-', linewidth=2, label='Moving avg')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Mean Reward')
axes[0].set_title('VB2 GRPO Training: Reward Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curve
if losses:
    loss_steps = [entry['step'] for entry in log_history if 'loss' in entry]
    axes[1].plot(loss_steps, losses, 'g-', alpha=0.7)
    axes[1].set_xlabel('Training Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training Loss')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vb2_reward_curve.png', dpi=150)
plt.show()
print(f"Final mean reward: {rewards[-1]:.4f}" if rewards else "No reward data yet")

## 8. Compare Before vs After

Run both the base and trained models on the same VB2 scenario.

In [ ]:
# Generate a test prompt
test_prompt = VB2_PROMPT_TEMPLATE.format(
    day=100, balance=450.0, weather="sunny", season="summer",
    machine_inv=json.dumps({p: 4 for p in PRODUCT_NAMES}),
    last_sales=json.dumps({"soda": 6, "water": 5, "candy": 3, "chips": 4, "sandwich": 2}),
    price_list="\n".join(f"  {p}: ${PRODUCTS[p]['ideal_price']:.2f}" for p in PRODUCT_NAMES),
)

text = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=128, temperature=0.3)
response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("Trained model output:")
print(response)
print()

# Evaluate
metrics = evaluate_pricing_decision(response, seed=100)
print(f"Evaluation: {metrics}")

## 9. Save Model

Save the fine-tuned model for deployment.

In [ ]:
# Save locally
if False:
    model.save_pretrained_merged("vb2_finetuned", tokenizer, save_method="merged_16bit")

# Push to HF Hub
if False:
    model.push_to_hub_merged("your-username/vb2-pricing-agent", tokenizer, save_method="merged_16bit", token="hf_...")

## Results

The GRPO training teaches the model to:
1. **Output valid JSON** pricing decisions (reward_valid_json)
2. **Set prices near optimal points** that maximize revenue (reward_revenue)
3. **Improve bank balance** by finding profitable price-demand equilibria (reward_balance)

This demonstrates that the VB2 environment produces meaningful training signal for RL-based agent improvement.